# VGG — Very Deep Convolutional Networks for Large-Scale Image Recognition (2014)

_Papers · Computer Vision_

**Go deep with nothing but stacked 3x3 convolutions: two 3x3 layers see as much as one 5x5, with fewer weights.**

---

This notebook is a practice scaffold for the **VGG — Very Deep Convolutional Networks for Large-Scale Image Recognition (2014)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torchvision, torchvision.transforms as T

torch.manual_seed(0)

# --- 0. The worked example: two stacked 3x3 convs vs one 5x5 conv, C=64 channels. ---
#     Same 5x5 receptive field; the stack uses fewer weights. (bias=False to match the paper's count.)
C = 64
two_3x3 = nn.Sequential(nn.Conv2d(C, C, 3, padding=1, bias=False),   # 9*C*C
                        nn.ReLU(),
                        nn.Conv2d(C, C, 3, padding=1, bias=False))   # 9*C*C
one_5x5 = nn.Conv2d(C, C, 5, padding=2, bias=False)                  # 25*C*C
p_two = sum(p.numel() for p in two_3x3.parameters())
p_one = sum(p.numel() for p in one_5x5.parameters())
print(f"two stacked 3x3 (C={C}): {p_two} weights   = 18*C^2 = {18*C*C}")
print(f"one      5x5    (C={C}): {p_one} weights   = 25*C^2 = {25*C*C}")
print(f"the 5x5 has {p_one - p_two} more weights ({p_one/p_two:.3f}x); both see a 5x5 receptive field")
# two stacked 3x3 (C=64): 73728 weights   = 18*C^2 = 73728
# one      5x5    (C=64): 102400 weights   = 25*C^2 = 102400
# the 5x5 has 28672 more weights (1.389x); both see a 5x5 receptive field


# --- 1. A VGG block: n [Conv3x3(pad 1) -> ReLU] then one 2x2 MaxPool. ---
def vgg_block(in_ch, out_ch, n_convs):
    layers = []
    for i in range(n_convs):
        layers += [nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True)]
    layers += [nn.MaxPool2d(2)]                 # 2x2, stride 2 -> halves H and W
    return nn.Sequential(*layers)


# --- 2. Stack blocks with growing channels into a small VGG-style net. ---
class SmallVGG(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            vgg_block(3,   32, 2),   # 32x32 -> 16x16   (two stacked 3x3, like VGG)
            vgg_block(32,  64, 2),   # 16x16 -> 8x8
            vgg_block(64, 128, 2))   #  8x8  -> 4x4
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Linear(256, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


# --- 3. A CIFAR-10 subset (torchvision is preinstalled in Colab). ---
tfm = T.Compose([T.ToTensor(),
                 T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
train_full = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=tfm)
test_full  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=tfm)
train_set  = torch.utils.data.Subset(train_full, range(5000))     # small + fast
test_set   = torch.utils.data.Subset(test_full,  range(2000))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256)
device = "cuda" if torch.cuda.is_available() else "cpu"

net = SmallVGG().to(device)
print("SmallVGG total parameters:", sum(p.numel() for p in net.parameters()))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
lf  = nn.CrossEntropyLoss()

for ep in range(5):
    net.train(); tot = 0.0; nb = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = lf(net(xb), yb); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    print(f"epoch {ep}  train loss {tot/nb:.4f}")

# --- 4. PRINT test accuracy. ---
net.eval(); correct = 0; total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = net(xb).argmax(1)
        correct += (pred == yb).sum().item(); total += yb.size(0)
print(f"\nTest accuracy on 2000 CIFAR-10 images: {100*correct/total:.1f}%")
# Well above the 10% chance baseline for 10 classes.
# (Exact number varies by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_For the same receptive field, how do parameters grow with filter size — and does the stacked-3x3 net actually learn on CIFAR?_

In [ ]:
import torch, torch.nn as nn

# --- Parameter counts for matched receptive fields (weights only, no bias), C=64. ---
C = 64
def conv_w(k, n=1):  # n stacked k x k convs, C->C, weights only
    return n * (k*k*C*C)
print("two   3x3 (5x5 RF):", conv_w(3, 2))   # 73728
print("one   5x5 (5x5 RF):", conv_w(5, 1))   # 102400
print("three 3x3 (7x7 RF):", conv_w(3, 3))   # 110592
print("one   7x7 (7x7 RF):", conv_w(7, 1))   # 200704

# --- The training-loss curve above came from the CODE cell's 5-epoch SmallVGG run
#     on a 5000-image CIFAR-10 subset (Adam, lr=1e-3). Numbers are ours, not the paper's. ---
# epoch 0  train loss 1.79
# epoch 1  train loss 1.36
# epoch 2  train loss 1.16
# epoch 3  train loss 1.00
# epoch 4  train loss 0.85

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The receptive-field / parameter trade. You want a stack whose outputs each see a $7\times7$
            patch of the input, with $C=128$ channels in and out. Compare building it from three $3\times3$
            convs vs one $7\times7$ conv: give the receptive field and the weight count of each (ignore bias),
            and say which the paper picks and why.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Receptive field: three stacked $3\times3$ convs reach $1 + 2\cdot3 = 7$ on a side, i.e. $7\times7$ &mdash; the same as one $7\times7$ conv. — _Each $3\times3$ conv adds radius 1; three of them add 3, so the reach is $7\times7$ (§2.3)._
- Weights of the stack: $3\,(3^2 C^2) = 27 C^2 = 27 \cdot 128^2 = 27 \cdot 16{,}384 = 442{,}368$. — _Each $3\times3$ layer is $9 C^2$ weights; three of them give $27 C^2$._
- Weights of the single $7\times7$: $7^2 C^2 = 49 C^2 = 49 \cdot 16{,}384 = 802{,}816$. — _One $7\times7$ layer is $49 C^2$ &mdash; the paper notes this is 81% more than $27 C^2$._
- Pick the stack. — _Same receptive field, ~45% fewer weights ($27/49$), and two extra ReLUs make it more expressive &mdash; the paper's two reasons._

**Answer:** Both reach a $7\times7$ receptive field. The three-$3\times3$ stack costs $27C^2 = 442{,}368$
                 weights; the single $7\times7$ costs $49C^2 = 802{,}816$ &mdash; 81% more. The paper picks the
                 stack: same reach, far fewer weights, and the extra non-linear ReLUs between the small layers
                 make the decision function more discriminative (§2.3).

</details>

**Problem 2.** The ablation. Take your working VGG-style net and replace each pair of stacked $3\times3$
            convs with one $5\times5$ conv (same in/out channels, padding 2 so the spatial size matches).
            What changes in (a) parameter count and (b) the number of non-linearities, and what does the swap
            cost the model?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Parameters: a pair of $3\times3$ ($18 C^2$) becomes one $5\times5$ ($25 C^2$), so each swapped block gains $7 C^2$ weights &mdash; the net gets bigger, not smaller. — _$25C^2 \gt 18C^2$: replacing the stack with one big filter raises the weight count (the paper's argument, run in reverse)._
- Non-linearities: the pair had two ReLUs (one after each conv); the single $5\times5$ has one. You lose a ReLU per swapped block. — _Fewer interleaved non-linearities make the layer's mapping closer to a single linear step, reducing expressiveness._
- Retrain on the same CIFAR subset and compare accuracy. — _An honest ablation changes one design choice &mdash; small-stacked vs one-large &mdash; so any accuracy/size difference is attributable to it._

**Answer:** Each $5\times5$ swap adds parameters ($25C^2$ vs $18C^2$ for the pair) and
                 removes one ReLU. So you get a larger, shallower-in-nonlinearity model &mdash; typically equal
                 or worse accuracy at higher cost. That is exactly why VGG keeps everything $3\times3$ and just
                 stacks more layers (§2.3). The CODE cell prints the $73{,}728$ vs $102{,}400$ counts that
                 underlie this.

</details>

**Problem 3.** Your parameter-count cell prints $73{,}728$ for two stacked $3\times3$ convs and $102{,}400$ for one
            $5\times5$ conv at $C=64$. A teammate rebuilds the convs with bias=True and now gets
            $73{,}856$ and $102{,}464$. Why did the numbers move, and which match the paper's formula?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Each conv layer adds $C$ bias terms: $64$ per layer. — _A conv with $C$ output channels has one bias per output channel, on top of the $k^2 C^2$ weights._
- Two $3\times3$ layers add $2\cdot64 = 128$: $73{,}728 + 128 = 73{,}856$. One $5\times5$ adds $64$: $102{,}400 + 64 = 102{,}464$. — _The bias counts are exactly the differences your teammate saw._
- Note the paper's $18C^2$ / $25C^2$ formula counts weights only, no bias. — _§2.3 compares $27C^2$ vs $49C^2$ with no bias term, so the clean numbers are $73{,}728$ and $102{,}400$._

**Answer:** The extra counts are the bias terms ($C=64$ per conv layer). With bias=True the
                 two-$3\times3$ stack gains $128$ ($\to 73{,}856$) and the $5\times5$ gains $64$ ($\to
                 102{,}464$). The paper's formula ($18C^2$ vs $25C^2$) is weights-only, so the figures that match
                 it &mdash; and the worked example &mdash; are $73{,}728$ and $102{,}400$, i.e. the
                 bias=False counts.

</details>